# FIGARO E3 local files

Read the 2015 JRC release under CC BY 4.0. Obtain CSVs and the classification workbook from https://data.jrc.ec.europa.eu/collection/id-00403. Replace the portable path in the three parse calls below. Both matrix and flat-file representations are supported; identical browser copies are collapsed and conflicting revisions raise.


In [ ]:
import mario


## Native supply-use system

The source has 46 regions, 176 activities and 213 commodities. Employment EDU and OCC are alternative estimates and remain separately labelled.


In [ ]:
db = mario.parse_figaro_e3(
    path="/path/to/FIGARO E3", table="SUT", year=2015,
    employment="both", input_format="auto", calc_all=False,
)


In [ ]:
{name: len(db.get_index(name)) for name in ["Region", "Activity", "Commodity", "Satellite account"]}


In [ ]:
db.info["figaro_e3"]["input_files"]


## Published emissions including direct final demand

The default CO2eq file already uses AR5/GWP100. Changing an IPCC argument does not recharacterise these rows. The gas-mass alternative keeps CO2, CH4 and N2O in kt and F-gases in kt CO2eq. Energy indicators overlap and EDU/OCC employment inventories must not be added together.


In [ ]:
db.calc_ghg(profile="figaro_e3", label="GHG source AR5 GWP100")
blocks = db.matrices["baseline"]
production_ghg = blocks["Ea"].loc["GHG source AR5 GWP100"].sum()
direct_ghg = blocks["EY"].loc["GHG source AR5 GWP100"].sum()
{"production_kt_CO2eq": float(production_ghg), "direct_final_demand_kt_CO2eq": float(direct_ghg)}


## Native industry IOT model D

The published industry IOT aligns directly with the industry-based extensions.


In [ ]:
industry = mario.parse_figaro_e3(
    path="/path/to/FIGARO E3", table="IOT", iot_mode="industry",
    year=2015, employment="both", calc_all=False,
)


In [ ]:
industry.matrices["baseline"]["Z"].shape


## Native product IOT model B

Read the monetary product IOT below. Industry-based extensions require explicit `extension_allocation="industry_technology"` and supply data. This derives product extensions from industry supply shares. Nonzero extensions on zero-output industries cannot be allocated and raise instead of being discarded; the SUT and industry IOT retain these native accounts.


In [ ]:
product = mario.parse_figaro_e3(
    path="/path/to/FIGARO E3", table="IOT", iot_mode="product",
    year=2015, extensions=False, calc_all=False,
)


In [ ]:
product.matrices["baseline"]["Z"].shape


## Interpretation and provenance

Native codes and signed primary-input/final-demand adjustments are preserved, including VY. Extension final-demand ordering differs from the economic tables and is matched by code. No standard FIGARO concordance is assumed. Supporting-document dates are revisions, while the observation year is 2015. The GAMS construction archive is not needed for CSV parsing. Methods: https://doi.org/10.1038/s41597-025-04431-z.
